# SynthAudit quickstart

This notebook validates the auditor against **planted ground truth**: a synthetic table whose artifacts we know exactly, because we put them there. It is the fastest way to see every module fire and to trust what the tool reports on real data.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd() if (Path.cwd() / 'datasets').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
from synthaudit import Audit, make_planted, score_detection

In [2]:
df, truth = make_planted(n=8000, seed=42)
df.head()

,x1,x2,x3,x4,x5,x6,x7,s1,s2,s3,...,regime_eff,thresh_flag,stab_score,sign_label,noise_col,const_col,dup_x1,post_outcome,leak_copy,target_y
0,0.304717,2.500403,2.954316,3.224131,2.842577,2.991118,-0.251693,2.234081,0.350631,1.048174,...,0.955133,0,-0.906268,stable,-0.113755,7.0,0.304717,3.475899,3.577042,3.577042
1,-1.039984,3.854515,0.680052,1.041683,1.656819,6.204141,-0.481334,0.175222,1.187826,0.311222,...,0.906938,0,-0.652552,stable,-0.107010,7.0,-1.039984,0.683137,-0.318109,-0.318110
2,0.750451,3.823744,1.190848,0.357984,3.803982,1.913820,0.201402,-0.051435,1.322728,1.277288,...,0.971293,0,-0.787606,stable,-0.190279,7.0,0.750451,3.295169,3.522101,3.522101
3,0.940565,1.323565,1.541778,0.541943,2.343168,6.085021,0.249360,0.846611,1.210562,-0.776911,...,0.953725,0,0.295270,unstable,2.252081,7.0,0.940565,2.973970,2.641438,2.641438
4,-1.951035,2.291026,1.956754,0.114450,3.760290,4.796257,0.156281,0.488886,-0.465944,-0.682638,...,0.928056,0,-0.983805,stable,0.731551,7.0,-1.951035,1.019702,0.359025,0.359025


The table plants ten artifacts: an exact linear identity, a four-variable balance constraint, a power law, a regime-affine mechanism, a threshold rule, a sign label, a constant, a jittered duplicate, an exact copy of the target, and a post-outcome feature. Honest inputs and a noise column serve as negative controls.

In [3]:
audit = Audit(df, target='target_y', name='quickstart')
results = audit.run()

[synthaudit] auditing 'quickstart' (8,000 rows x 24 cols, target=target_y)


[synthaudit] profile: 22 numeric, 3 categorical, 0 duplicate rows


[synthaudit] identity mining: 8 exact, 2 near identities, 0 FDs, 1 rule-derived labels


[synthaudit] determinism sweep: 2 deterministic, 9 near-deterministic columns


[synthaudit] causal scan: 7 edges on stochastic core (7 columns excluded)


[synthaudit] leakage audit: 5 critical, 1 high findings


[synthaudit] BTI = 0.217 (grade F) pillars={'L': 0.0, 'F': 0.5217, 'H': 1.0, 'R': 0.9697, 'I': 0.9286}


[synthaudit] done in 25.68s


In [4]:
sc = score_detection(results, truth)
print(f"planted artifacts detected: {sc['n_detected']}/{sc['n_planted']}")
print(f"negative controls clean:   {sc['negative_control_pass']:.0%}")

planted artifacts detected: 10/10
negative controls clean:   100%


In [5]:
for f in results['identity']['identities']:
    print(f"{f['type']:22s} {f.get('equation','')[:74]}")

duplicate_column       dup_x1 = 1*x1
duplicate_column       leak_copy = 1*target_y - 3.93131e-09
regime_affine          regime_eff = -0.015*x6 + offset(regime)
linear                 lin_exact = 3*x1 -2*x2 + 5
linear                 s4 = -1*s1 -1*s2 -1*s3
power_law              power_law = 2.5 * x3^2 * x4^1 * x5^-0.5
linear                 post_outcome = 0.15*x6 +0.85*target_y - 0.0001
linear                 target_y = -0.1764*x6 +1.1755*post_outcome + 0.0015
threshold_derivation   sign_label == 'stable'  iff  stab_score <= 4.36087e-05


In [6]:
results['scoring']

{'pillars': {'L': 0.0, 'F': 0.5217, 'H': 1.0, 'R': 0.9697, 'I': 0.9286},
 'weights': {'L': 0.3, 'F': 0.2, 'H': 0.2, 'R': 0.15, 'I': 0.15},
 'bti': 0.2171,
 'grade': 'F',
 'evidence': {'trivial_mechanism_score': 1.0,
  'label_evidence': ['leak_copy = 1*target_y - 3.93131e-09',
   'post_outcome = 0.15*x6 +0.85*target_y - 0.0001',
   'target_y = -0.1764*x6 +1.1755*post_outcome + 0.0015'],
  'artifact_columns': 11,
  'honest_model_score': 0.9696,
  'honest_baseline': 0.0,
  'honest_features_used': 12},
 'interpretation': 'BTI is a triage score: report the pillar vector, not just the scalar. Grade A/B: usable with the recommended feature view. C: quarantine flagged columns and re-benchmark. D/F: dataset measures generator recovery, not learning — unsuitable as an ML benchmark without redesign.'}

Grade F is the correct verdict here: the table ships a copy of its own target. On your own data the workflow is identical: `Audit(df, target=...).run()` and read the report.

In [7]:
audit.generate_report('quickstart_report.html')
print('wrote quickstart_report.html')

wrote quickstart_report.html
